# Notebook 02 — Dimension Tables

Build all external dimension tables saved to `data/external/`.

| Table | Source | Status |
|-------|--------|--------|
| `dim_calendar.csv` | `holidays` library + custom logic | ☐ |
| `dim_weather.csv` | Open-Meteo API | ☐ |
| `dim_events.csv` | Simulated CSV | ☐ |
| `dim_energy.csv` | GME PUN real data + 2026 scenario | ☐ |

## 1 — dim_calendar

**Source:** `holidays` library (Italy subdiv='CO' + Switzerland cantons TI/GR/ZH/BE) + custom ponte logic + meteorological seasons.

**Output:** `data/external/dim_calendar.csv`  
**Columns:** `date | is_weekend | is_holiday | is_ponte | is_swiss_holiday | season`

In [1]:
import sys
import pandas as pd
import holidays
from datetime import timedelta
from pathlib import Path

# Add project root to sys.path so 'src' is importable from any CWD
_project_root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'src').exists())
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import src.utils_forecast as ut_f


# ── Configuration ──────────────────────────────────────────────────────────────
YEARS = [2023, 2024]
DATE_RANGE = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
SWISS_CANTONS = ['TI', 'GR', 'ZH', 'BE']

OUTPUT_PATH = Path('../data/external/dim_calendar.csv')

# ── Italian holidays: national + Como (subdiv='CO') ───────────────────────────
it_holidays = holidays.Italy(subdiv='CO', years=YEARS)
it_holidays_set = set(it_holidays.keys())

# ── Swiss holidays: union of 4 cantons ────────────────────────────────────────
swiss_holidays_set = set()
for canton in SWISS_CANTONS:
    h = holidays.Switzerland(subdiv=canton, years=YEARS)
    swiss_holidays_set.update(h.keys())

print(f"Italian holidays (CO): {len(it_holidays_set)} dates")
print(f"Swiss holidays (TI+GR+ZH+BE): {len(swiss_holidays_set)} dates")

Italian holidays (CO): 28 dates
Swiss holidays (TI+GR+ZH+BE): 38 dates


In [2]:
ponte_set = ut_f.compute_ponte(DATE_RANGE, it_holidays_set)
print(f"Ponte days detected: {len(ponte_set)}")
print("Ponte dates:")
for d in sorted(ponte_set):
    neighbor_info = []
    for delta in [-2, -1, 1, 2]:
        n = d + timedelta(days=delta)
        tag = []
        if n in it_holidays_set: tag.append('HOL')
        if n.weekday() >= 5:     tag.append('WKD')
        if tag:
            neighbor_info.append(f"{n}({'|'.join(tag)})")
    print(f"  {d} ({d.strftime('%A')}) ← neighbors: {', '.join(neighbor_info)}")


Ponte days detected: 6
Ponte dates:
  2023-04-24 (Monday) ← neighbors: 2023-04-22(WKD), 2023-04-23(WKD), 2023-04-25(HOL)
  2023-08-14 (Monday) ← neighbors: 2023-08-12(WKD), 2023-08-13(WKD), 2023-08-15(HOL)
  2023-09-01 (Friday) ← neighbors: 2023-08-31(HOL), 2023-09-02(WKD), 2023-09-03(WKD)
  2024-04-26 (Friday) ← neighbors: 2024-04-25(HOL), 2024-04-27(WKD), 2024-04-28(WKD)
  2024-08-16 (Friday) ← neighbors: 2024-08-15(HOL), 2024-08-17(WKD), 2024-08-18(WKD)
  2024-12-27 (Friday) ← neighbors: 2024-12-25(HOL), 2024-12-26(HOL), 2024-12-28(WKD), 2024-12-29(WKD)


In [3]:
# ── Build dim_calendar ─────────────────────────────────────────────────────────
dim_calendar = pd.DataFrame({'date': DATE_RANGE})
dim_calendar['date'] = dim_calendar['date'].dt.date

dim_calendar['is_weekend']       = dim_calendar['date'].apply(lambda d: d.weekday() >= 5)
dim_calendar['is_holiday']       = dim_calendar['date'].apply(lambda d: d in it_holidays_set)
dim_calendar['is_ponte']         = dim_calendar['date'].apply(lambda d: d in ponte_set)
dim_calendar['is_swiss_holiday'] = dim_calendar['date'].apply(lambda d: d in swiss_holidays_set)
dim_calendar['season']           = dim_calendar['date'].apply(lambda d: ut_f.get_season(d.month))

# is_high_season: placeholder Jun–Sep, to be revised after join with raw_sales_pos
# Real boundaries will be derived from daily covers distribution in Notebook 03.
dim_calendar['is_high_season']   = dim_calendar['date'].apply(lambda d: d.month in [6, 7, 8, 9])

# ── Sanity checks ──────────────────────────────────────────────────────────────
print(f"Shape: {dim_calendar.shape}")
print(f"\nValue counts — is_holiday      : {dim_calendar['is_holiday'].sum()} days")
print(f"Value counts — is_ponte        : {dim_calendar['is_ponte'].sum()} days")
print(f"Value counts — is_weekend      : {dim_calendar['is_weekend'].sum()} days")
print(f"Value counts — is_swiss_holiday: {dim_calendar['is_swiss_holiday'].sum()} days")
print(f"Value counts — is_high_season  : {dim_calendar['is_high_season'].sum()} days")
print(f"\nSeason distribution:\n{dim_calendar['season'].value_counts().sort_index()}")
print("\nFirst 10 rows:")
display(dim_calendar.head(10))

Shape: (731, 7)

Value counts — is_holiday      : 28 days
Value counts — is_ponte        : 6 days
Value counts — is_weekend      : 209 days
Value counts — is_swiss_holiday: 38 days
Value counts — is_high_season  : 244 days

Season distribution:
season
autumn    182
spring    184
summer    184
winter    181
Name: count, dtype: int64

First 10 rows:


,date,is_weekend,is_holiday,is_ponte,is_swiss_holiday,season,is_high_season
0,2023-01-01,True,True,False,True,winter,False
1,2023-01-02,False,False,False,True,winter,False
2,2023-01-03,False,False,False,False,winter,False
3,2023-01-04,False,False,False,False,winter,False
4,2023-01-05,False,False,False,False,winter,False
5,2023-01-06,False,True,False,True,winter,False
6,2023-01-07,True,False,False,False,winter,False
7,2023-01-08,True,False,False,False,winter,False
8,2023-01-09,False,False,False,False,winter,False
9,2023-01-10,False,False,False,False,winter,False


In [4]:
# ── AWAIT GIOVANNI APPROVAL before saving ─────────────────────────────────────
# Review the output above (shape, ponte dates, counts) before running this cell.

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
dim_calendar.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH.resolve()}")

Saved: /Users/giovannilopresti/Documents/Data Analytics/Progetti Risto/horeca_forecasting/data/external/dim_calendar.csv


## 2 — dim_weather

**Source:** Open-Meteo historical archive API — free, no API key required.  
**Coordinates:** Como (45.81°N, 9.09°E)  
**Variables:** `temperature_2m_max`, `temperature_2m_min`, `precipitation_sum`  
**Output:** `data/external/dim_weather.csv`  
**Columns:** `date | avg_temp | rain_mm | is_bad_weather`

In [5]:
import requests
import pandas as pd
from pathlib import Path

# ── API call ───────────────────────────────────────────────────────────────────
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 45.81,
    "longitude": 9.09,
    "start_date": "2023-01-01",
    "end_date": "2024-12-31",
    "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"],
    "timezone": "Europe/Rome"
}

response = requests.get(url, params=params)
response.raise_for_status()
data = response.json()

# ── Build DataFrame ────────────────────────────────────────────────────────────
daily = data["daily"]
weather_raw = pd.DataFrame({
    "date":     pd.to_datetime(daily["time"]).date,
    "temp_max": daily["temperature_2m_max"],
    "temp_min": daily["temperature_2m_min"],
    "rain_mm":  daily["precipitation_sum"],
})

weather_raw["avg_temp"] = ((weather_raw["temp_max"] + weather_raw["temp_min"]) / 2).round(1)
weather_raw = weather_raw.drop(columns=["temp_max", "temp_min"])

print(f"API returned: {len(weather_raw)} rows")
print(f"Date range  : {weather_raw['date'].min()} → {weather_raw['date'].max()}")
print(f"NaN count   :\n{weather_raw.isna().sum()}")

API returned: 731 rows
Date range  : 2023-01-01 → 2024-12-31
NaN count   :
date        0
rain_mm     0
avg_temp    0
dtype: int64


In [6]:
# ── Join on dim_calendar — gap check ──────────────────────────────────────────
dim_calendar = pd.read_csv('../data/external/dim_calendar.csv', parse_dates=['date'])
dim_calendar['date'] = dim_calendar['date'].dt.date

weather_raw['date'] = pd.to_datetime(weather_raw['date']).dt.date

merged = dim_calendar[['date']].merge(weather_raw, on='date', how='left')

gaps = merged[merged['avg_temp'].isna()]

print(f"dim_calendar rows : {len(dim_calendar)}")
print(f"weather API rows  : {len(weather_raw)}")
print(f"After LEFT JOIN   : {len(merged)}")
print(f"Gap dates (NaN)   : {len(gaps)}")

if len(gaps) == 0:
    print("✓ No gaps — all dates covered.")
else:
    print("⚠ Missing dates in weather data:")
    print(gaps['date'].tolist())

dim_calendar rows : 731
weather API rows  : 731
After LEFT JOIN   : 731
Gap dates (NaN)   : 0
✓ No gaps — all dates covered.


In [7]:
# ── Distribution — is_bad_weather threshold choice ────────────────────────────
print("── avg_temp distribution ──")
print(merged['avg_temp'].describe().round(1))
print(f"\nDays below 5°C  : {(merged['avg_temp'] < 5).sum()}")
print(f"Days below 0°C  : {(merged['avg_temp'] < 0).sum()}")

print("\n── rain_mm distribution ──")
print(merged['rain_mm'].describe().round(1))
print(f"\nDays with rain >  0 mm: {(merged['rain_mm'] > 0).sum()}")
print(f"Days with rain >  5 mm: {(merged['rain_mm'] > 5).sum()}")
print(f"Days with rain > 10 mm: {(merged['rain_mm'] > 10).sum()}")
print(f"Days with rain > 20 mm: {(merged['rain_mm'] > 20).sum()}")

# Threshold confirmed: rain_mm > 10 OR avg_temp < 5
# rain > 10mm = heavy rain deterrent (pre-alpine convective pattern, peaks May/Oct)
# temp < 5°C  = cold deterrent (winter complement, 46 days)

# ── Build final dim_weather ────────────────────────────────────────────────────
dim_weather = merged[['date', 'avg_temp', 'rain_mm']].copy()
dim_weather['is_bad_weather'] = (dim_weather['rain_mm'] > 10) | (dim_weather['avg_temp'] < 5)

print(f"\nis_bad_weather days: {dim_weather['is_bad_weather'].sum()} ({dim_weather['is_bad_weather'].mean()*100:.1f}%)")
print(f"\nFirst 10 rows:")
display(dim_weather.head(10))

── avg_temp distribution ──
count    731.0
mean      14.6
std        6.9
min        0.9
25%        8.8
50%       14.0
75%       20.4
max       30.0
Name: avg_temp, dtype: float64

Days below 5°C  : 46
Days below 0°C  : 0

── rain_mm distribution ──
count    731.0
mean       5.0
std       11.6
min        0.0
25%        0.0
50%        0.2
75%        4.2
max      104.8
Name: rain_mm, dtype: float64

Days with rain >  0 mm: 401
Days with rain >  5 mm: 162
Days with rain > 10 mm: 108
Days with rain > 20 mm: 57

is_bad_weather days: 154 (21.1%)

First 10 rows:


,date,avg_temp,rain_mm,is_bad_weather
0,2023-01-01,9.8,2.2,False
1,2023-01-02,9.8,2.5,False
2,2023-01-03,9.8,4.4,False
3,2023-01-04,8.8,0.0,False
4,2023-01-05,9.1,0.0,False
5,2023-01-06,8.0,0.0,False
6,2023-01-07,6.6,0.1,False
7,2023-01-08,7.0,23.7,True
8,2023-01-09,9.9,2.5,False
9,2023-01-10,8.7,0.0,False


In [8]:
OUTPUT_WEATHER = Path('../data/external/dim_weather.csv')
dim_weather.to_csv(OUTPUT_WEATHER, index=False)
print(f"Saved: {OUTPUT_WEATHER.resolve()}")
print(f"Shape: {dim_weather.shape}")

Saved: /Users/giovannilopresti/Documents/Data Analytics/Progetti Risto/horeca_forecasting/data/external/dim_weather.csv
Shape: (731, 4)


## 3 — dim_events

**Source:** Real 2023-2024 events scraped from Wikipedia, official club/venue sites + manual entries.  
**Note on quality:** dates marked `# EST` are estimated from historical patterns — verify and correct if needed.  
**Output:** `data/external/dim_events.csv`  
**Columns:** `date | city | event_name | event_type | magnitude | radius_km`

In [9]:
events = []

def single(date):
    return [date]

def add(date_list, city, name, etype, magnitude, radius_km):
    for d in date_list:
        events.append({
            "date": d, "city": city, "event_name": name,
            "event_type": etype, "magnitude": magnitude, "radius_km": radius_km
        })

# ══════════════════════════════════════════════════════════════════════════════
# COMO — LOCAL & PROVINCIAL EVENTS
# ══════════════════════════════════════════════════════════════════════════════

# ── Como 1907 — Serie B home matches 2023-24 (source: Wikipedia) ──────────────
serie_b = [
    ("2023-08-26", "vs Reggiana"),    ("2023-09-17", "vs Ternana"),
    ("2023-09-27", "vs Sampdoria"),   ("2023-10-28", "vs Catanzaro"),
    ("2023-11-25", "vs Feralpisalò"), ("2023-11-28", "vs Lecco"),
    ("2023-12-10", "vs Modena"),      ("2023-12-23", "vs Palermo"),
    ("2024-01-13", "vs Spezia"),      ("2024-01-27", "vs Ascoli"),
    ("2024-02-09", "vs Brescia"),     ("2024-02-24", "vs Parma"),
    ("2024-03-03", "vs Venezia"),     ("2024-03-16", "vs Pisa"),
    ("2024-04-01", "vs Südtirol"),    ("2024-04-13", "vs Bari"),
    ("2024-05-01", "vs Cittadella"),  ("2024-05-10", "vs Cosenza"),
]
for date, opp in serie_b:
    add(single(date), "Como", f"Como 1907 {opp} (Serie B)",
        "sports", magnitude=1, radius_km=2)

# ── Como 1907 — Serie A home matches 2024-25 through Dec 2024 (source: Wikipedia) ──
serie_a = [
    ("2024-09-14", "vs Bologna"),      ("2024-09-29", "vs Hellas Verona"),
    ("2024-10-19", "vs Parma"),        ("2024-10-31", "vs Lazio"),
    ("2024-11-24", "vs Fiorentina"),   ("2024-11-30", "vs Monza"),
]
for date, opp in serie_a:
    add(single(date), "Como", f"Como 1907 {opp} (Serie A)",
        "sports", magnitude=2, radius_km=2)

# ── Fiera di Sant'Abbondio (source: confirmed by user — 31 agosto ogni anno) ──
add(single("2023-08-31"), "Como", "Fiera di Sant'Abbondio", "festival", 2, 3)
add(single("2024-08-31"), "Como", "Fiera di Sant'Abbondio", "festival", 2, 3)

# ── Palio del Baradello — EST: last weekend of September ─────────────────────
add(ut_f.days("2023-09-23", "2023-09-24"), "Como", "Palio del Baradello", "festival", 2, 3)  # EST
add(ut_f.days("2024-09-21", "2024-09-22"), "Como", "Palio del Baradello", "festival", 2, 3)  # EST

# ── Mercatini di Natale Como (December 1-26 both years) ──────────────────────
add(ut_f.days("2023-12-01", "2023-12-26"), "Como", "Mercatini di Natale Como", "market", 2, 3)
add(ut_f.days("2024-12-01", "2024-12-26"), "Como", "Mercatini di Natale Como", "market", 2, 3)

# ── Teatro Sociale Como — major shows only (source: teatrosocialecomo.it) ─────
teatro = [
    ("2023-09-28", "2023-09-30", "Die Zauberflöte (Mozart) — Teatro Sociale"),
    ("2023-10-03", "2023-10-03", "Carmina Burana — Teatro Sociale"),
    ("2023-11-08", "2023-11-09", "La vita davanti a sé — Teatro Sociale"),
    ("2023-12-08", "2023-12-10", "Don Carlo (Verdi) — Teatro Sociale"),
    ("2024-03-24", "2024-03-24", "Sister Act (musical) — Teatro Sociale"),
    ("2024-04-10", "2024-04-11", "Grease (musical) — Teatro Sociale"),
]
for start, end, name in teatro:
    add(ut_f.days(start, end), "Como", name, "concert", 1, 2)

# ══════════════════════════════════════════════════════════════════════════════
# CERNOBBIO — VILLA D'ESTE (radius ~7km from Como centre)
# ══════════════════════════════════════════════════════════════════════════════

# ── Concorso d'Eleganza Villa d'Este (source: Wikipedia) ─────────────────────
add(ut_f.days("2023-05-19", "2023-05-21"), "Cernobbio", "Concorso d'Eleganza Villa d'Este", "festival", 3, 7)
add(ut_f.days("2024-05-24", "2024-05-26"), "Cernobbio", "Concorso d'Eleganza Villa d'Este", "festival", 3, 7)

# ── Orticolario — garden & landscape fair (source: provided by Giovanni) ──────
add(ut_f.days("2023-10-05", "2023-10-08"), "Cernobbio", "Orticolario 2023", "fair", 2, 7)
add(ut_f.days("2024-10-03", "2024-10-06"), "Cernobbio", "Orticolario 2024", "fair", 2, 7)

# ── Proposte — horeca trade fair (source: provided by Giovanni) ───────────────
add(ut_f.days("2023-10-14", "2023-10-17"), "Cernobbio", "Proposte 2023", "fair", 2, 7)
add(ut_f.days("2024-10-12", "2024-10-15"), "Cernobbio", "Proposte 2024", "fair", 2, 7)

# ══════════════════════════════════════════════════════════════════════════════
# COMO AREA — TOURISM PEAKS (radius 0 = applies to entire lake area)
# ══════════════════════════════════════════════════════════════════════════════

# ── Ferragosto tourism peak (Aug 10-18 both years) ───────────────────────────
add(ut_f.days("2023-08-10", "2023-08-18"), "Como", "Ferragosto — Tourism Peak", "tourism_peak", 3, 0)
add(ut_f.days("2024-08-10", "2024-08-18"), "Como", "Ferragosto — Tourism Peak", "tourism_peak", 3, 0)

# ── Pasqua + Pasquetta (Easter 2023: Apr 9-10 | Easter 2024: Mar 31 - Apr 1) ─
add(ut_f.days("2023-04-08", "2023-04-10"), "Como", "Pasqua e Pasquetta", "tourism_peak", 2, 0)
add(ut_f.days("2024-03-30", "2024-04-01"), "Como", "Pasqua e Pasquetta", "tourism_peak", 2, 0)

# ── Ponte Capodanno (Dec 30 - Jan 2, only dates within 2023-2024 range) ──────
add(ut_f.days("2023-01-01", "2023-01-02"), "Como", "Ponte Capodanno", "tourism_peak", 2, 0)
add(ut_f.days("2023-12-30", "2024-01-02"), "Como", "Ponte Capodanno", "tourism_peak", 2, 0)

# ══════════════════════════════════════════════════════════════════════════════
# MONZA (radius ~45km from Como centre)
# ══════════════════════════════════════════════════════════════════════════════

# ── GP Italia F1 Monza (source: Wikipedia) ────────────────────────────────────
add(ut_f.days("2023-09-01", "2023-09-03"), "Monza", "GP Italia F1", "sports", 3, 45)
add(ut_f.days("2024-08-30", "2024-09-01"), "Monza", "GP Italia F1", "sports", 3, 45)

# ══════════════════════════════════════════════════════════════════════════════
# MILANO (radius ~50km from Como centre)
# ══════════════════════════════════════════════════════════════════════════════

# ── Salone del Mobile (source: Wikipedia) ─────────────────────────────────────
add(ut_f.days("2023-04-18", "2023-04-23"), "Milano", "Salone del Mobile 2023", "fair", 3, 50)
add(ut_f.days("2024-04-16", "2024-04-21"), "Milano", "Salone del Mobile 2024", "fair", 3, 50)

# ── EICMA (source: Wikipedia 2023 confirmed | 2024 EST pattern) ──────────────
add(ut_f.days("2023-11-07", "2023-11-12"), "Milano", "EICMA 2023", "fair", 3, 50)
add(ut_f.days("2024-11-05", "2024-11-10"), "Milano", "EICMA 2024", "fair", 3, 50)  # EST

# ── Milano Fashion Week (EST: typical Feb/Sep dates ±1-2 days) ───────────────
add(ut_f.days("2023-02-21", "2023-02-27"), "Milano", "Milano Fashion Week Feb 2023", "fair", 3, 50)  # EST
add(ut_f.days("2023-09-19", "2023-09-25"), "Milano", "Milano Fashion Week Sep 2023", "fair", 3, 50)  # EST
add(ut_f.days("2024-02-20", "2024-02-26"), "Milano", "Milano Fashion Week Feb 2024", "fair", 3, 50)  # EST
add(ut_f.days("2024-09-17", "2024-09-23"), "Milano", "Milano Fashion Week Sep 2024", "fair", 3, 50)  # EST

# ══════════════════════════════════════════════════════════════════════════════
# BUILD DATAFRAME
# ══════════════════════════════════════════════════════════════════════════════
dim_events = pd.DataFrame(events)
dim_events['date'] = pd.to_datetime(dim_events['date']).dt.date

# Keep only dates within our 2023-2024 range
start_date = pd.Timestamp("2023-01-01").date()
end_date   = pd.Timestamp("2024-12-31").date()
dim_events = dim_events[(dim_events['date'] >= start_date) & (dim_events['date'] <= end_date)]
dim_events = dim_events.sort_values('date').reset_index(drop=True)

print(f"Shape: {dim_events.shape}")
print(f"Date range: {dim_events['date'].min()} → {dim_events['date'].max()}")
print(f"\nevent_type counts:\n{dim_events['event_type'].value_counts()}")
print(f"\ncity counts:\n{dim_events['city'].value_counts()}")
print(f"\nmagnitude counts:\n{dim_events['magnitude'].value_counts().sort_index()}")
print(f"\nUnique event days (distinct dates with ≥1 event): {dim_events['date'].nunique()}")
print("\nSample — first 15 rows:")
display(dim_events.head(15))

Shape: (204, 6)
Date range: 2023-01-01 → 2024-12-26

event_type counts:
event_type
fair            68
market          52
tourism_peak    30
sports          30
festival        12
concert         12
Name: count, dtype: int64

city counts:
city
Como         124
Milano        52
Cernobbio     22
Monza          6
Name: count, dtype: int64

magnitude counts:
magnitude
1    30
2    92
3    82
Name: count, dtype: int64

Unique event days (distinct dates with ≥1 event): 190

Sample — first 15 rows:


,date,city,event_name,event_type,magnitude,radius_km
0,2023-01-01,Como,Ponte Capodanno,tourism_peak,2,0
1,2023-01-02,Como,Ponte Capodanno,tourism_peak,2,0
2,2023-02-21,Milano,Milano Fashion Week Feb 2023,fair,3,50
3,2023-02-22,Milano,Milano Fashion Week Feb 2023,fair,3,50
4,2023-02-23,Milano,Milano Fashion Week Feb 2023,fair,3,50
5,2023-02-24,Milano,Milano Fashion Week Feb 2023,fair,3,50
6,2023-02-25,Milano,Milano Fashion Week Feb 2023,fair,3,50
7,2023-02-26,Milano,Milano Fashion Week Feb 2023,fair,3,50
8,2023-02-27,Milano,Milano Fashion Week Feb 2023,fair,3,50
9,2023-04-08,Como,Pasqua e Pasquetta,tourism_peak,2,0


In [10]:
OUTPUT_EVENTS = Path('../data/external/dim_events.csv')
dim_events.to_csv(OUTPUT_EVENTS, index=False)
print(f"Saved: {OUTPUT_EVENTS.resolve()}")
print(f"Shape: {dim_events.shape}")

Saved: /Users/giovannilopresti/Documents/Data Analytics/Progetti Risto/horeca_forecasting/data/external/dim_events.csv
Shape: (204, 6)
